# Questão 3
## 1. Motivação
Como a participação de candidatas e de eleitas varia entre partidos, estados e municípios?

Como as bancadas e as câmaras são pequenas (as menores bancadas de deputado federal têm 8 cadeiras, e a câmara municipal mediana tem 9 vereadores), um percentual de eleitas calculado por unidade é instável: uma pessoa a mais ou a menos muda o resultado em mais de 10 pontos percentuais. Por isso, traduzimos a pergunta em três:

1. **Partidos:** em 2022 (deputados federais), qual a amplitude do % de candidatas e de eleitas entre as federações?
2. **Estados:** como o % de candidatas e de eleitas varia entre as UFs e ao longo do tempo?
3. **Municípios:** que fração das câmaras municipais não tem nenhuma vereadora, e como isso evolui?

## 2. Hipótese

Esperamos que as candidaturas femininas sejam parecidas entre os partidos, porque a cota fixa um piso comum para todas as listas, e que a fração de eleitas varie mais, porque o resultado depende da votação de cada candidata. Esperamos também que, na maioria dos partidos e estados, a fração de eleitas fique abaixo da de candidatas. Por fim, esperamos que a fração de câmaras municipais sem nenhuma vereadora caia ao longo do tempo, acompanhando o aumento das candidaturas.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"

# dados: candidaturas ligadas aos resultados (2002-2024)
df = pd.read_parquet("../data/resultados_candidato.parquet")

# eleito: inclui eleito, eleito por qp e eleito por media (suplentes não contam)
df["eleito"] = df["resultado"].fillna("").str.startswith("eleito")

# presidente em 2006 e 2010: o resultado falta na base; vale o mais votado do último turno
pres = df[(df["cargo"] == "presidente") & (df["tipo_eleicao"] == "eleicao ordinaria") & df["turno"].notna()].copy()
pres = pres[pres["turno"] == pres.groupby("ano")["turno"].transform("max")]
venc = pres.loc[pres.groupby("ano")["votos"].idxmax()]
df.loc[venc.index, "eleito"] = True

# senador MT 2018: Selma Arruda (PSL), eleita e cassada depois, aparece sem resultado e sem votos
mt18 = ((df["cargo"] == "senador") & (df["ano"] == 2018) &
        (df["sigla_uf"] == "MT") & (df["tipo_eleicao"] == "eleicao ordinaria"))
df.loc[mt18, "eleito"] = df.loc[mt18, "resultado"].fillna("").str.startswith("eleito")
selma = mt18 & (df["sigla_partido"] == "PSL") & (df["genero"] == "feminino")
df.loc[selma, "eleito"] = True

# recorte, o mesmo das questões 1 e 2
base = df[df["tipo_eleicao"] == "eleicao ordinaria"].copy()
base = base[base["genero"].isin(["feminino", "masculino"])]

proporcionais = ["vereador", "deputado estadual", "deputado federal", "deputado distrital"]
majoritarios  = ["prefeito", "governador", "senador", "presidente"]
base = base[base["cargo"].isin(proporcionais + majoritarios)]

invalidas = ["indeferido", "indeferido com recurso", "renuncia", "cancelado",
             "cancelado com recurso", "cassado", "cassado com recurso",
             "falecido", "falecimento", "inelegivel", "impugnado",
             "nao conhecimento do pedido", "pedido nao conhecido",
             "pedido nao conhecido com recurso"]
base["valida"] = ~base["situacao"].isin(invalidas)
base["valida"] = base["valida"] | base["eleito"]    # quem foi eleito conta como candidatura válida


def pct_fem(d, chaves):
    t = d.groupby(chaves + ["genero"], observed=True).size().unstack("genero", fill_value=0)
    t["total"] = t["feminino"] + t["masculino"]
    t["pct_fem"] = t["feminino"] / t["total"] * 100
    return t.reset_index()


def cand_eleit(d, chaves):
    """Candidatas e eleitas lado a lado para qualquer agrupamento."""
    c = pct_fem(d, chaves)
    e = pct_fem(d[d["eleito"]], chaves)
    t = c.merge(e, on=chaves, how="left", suffixes=("_cand", "_eleit"))
    t["feminino_eleit"] = t["feminino_eleit"].fillna(0)
    t["total_eleit"] = t["total_eleit"].fillna(0)
    t["pct_fem_eleit"] = t["feminino_eleit"] / t["total_eleit"] * 100   # NaN se ninguém foi eleito
    return t

In [2]:
FEDERACOES_2022 = {
    "PT": "Fe Brasil (PT/PCdoB/PV)", "PC do B": "Fe Brasil (PT/PCdoB/PV)", "PV": "Fe Brasil (PT/PCdoB/PV)",
    "PSDB": "PSDB Cidadania", "CIDADANIA": "PSDB Cidadania",
    "PSOL": "PSOL Rede", "REDE": "PSOL Rede",
}

base["sigla_partido"] = base["sigla_partido"].astype(str)
base["unidade"] = base["sigla_partido"]
m22 = base["ano"] == 2022
base.loc[m22, "unidade"] = base.loc[m22, "sigla_partido"].map(FEDERACOES_2022).fillna(base.loc[m22, "sigla_partido"])

In [3]:
base[m22 & (base["cargo"] == "deputado federal")]["unidade"].value_counts().head(10)

unidade
Fe Brasil (PT/PCdoB/PV)    532
REPUBLICANOS               531
UNIÃO                      523
PTB                        512
PL                         509
PP                         508
PODE                       506
PSOL Rede                  496
MDB                        490
PSDB Cidadania             487
Name: count, dtype: int64

In [4]:
print(df.shape, base.shape)

chk = base[base["eleito"]].groupby(["ano", "cargo"]).size().unstack("cargo")
chk[["deputado federal", "senador", "governador", "presidente"]]

(2925021, 13) (2814082, 15)


cargo,deputado federal,senador,governador,presidente
ano,,,,
2002,513.0,54.0,27.0,1.0
2004,NaN,NaN,NaN,NaN
2006,513.0,27.0,27.0,1.0
2008,NaN,NaN,NaN,NaN
2010,513.0,54.0,27.0,1.0
2012,NaN,NaN,NaN,NaN
2014,513.0,27.0,27.0,1.0
2016,NaN,NaN,NaN,NaN
2018,513.0,54.0,27.0,1.0


In [5]:
base[(base["cargo"] == "senador") & (base["ano"] == 2018) & base["eleito"]]["genero"].value_counts()

genero
masculino    47
feminino      7
Name: count, dtype: int64

## 3. Partidos e federações

Em 2022, partidos que formaram federações passaram a disputar as eleições em conjunto. Por isso, nessa eleição tratamos como uma única unidade a Federação Brasil da Esperança (PT, PCdoB e PV), PSDB Cidadania e PSOL Rede. Nas demais eleições, cada partido é uma unidade. Para a fração de candidatas, só entram unidades com pelo menos 100 candidaturas; para a de eleitas, pelo menos 10 eleitos. Abaixo desses números, o percentual muda muito com uma ou duas pessoas.

In [6]:
N_MIN_CAND = 100     # partidos com menos candidaturas ficam de fora
N_MIN_ELEIT = 10     # o % de eleitas só aparece com pelo menos 10 eleitos

alvo = base[(base["ano"] == 2022) & (base["cargo"] == "deputado federal")]

part = cand_eleit(alvo, ["unidade"])
n_antes = len(part)
part = part[part["total_cand"] >= N_MIN_CAND].copy()
print(f"{len(part)} de {n_antes} partidos com {N_MIN_CAND} ou mais candidaturas")

# tipos explícitos (evita figura em branco no Plotly)
part["unidade"] = part["unidade"].astype(str)
for c in ["total_cand", "total_eleit", "feminino_eleit"]:
    part[c] = part[c].astype(int)
for c in ["pct_fem_cand", "pct_fem_eleit"]:
    part[c] = part[c].astype(float)

part["eleitas_confiavel"] = part["total_eleit"] >= N_MIN_ELEIT
part.loc[~part["eleitas_confiavel"], "pct_fem_eleit"] = np.nan
part = part.sort_values("pct_fem_cand")

# média do país, calculada a partir dos dados (e não escrita à mão)
br = cand_eleit(alvo.assign(pais="Brasil"), ["pais"])
pct_br_cand = br["pct_fem_cand"].iloc[0]
pct_br_eleit = br["pct_fem_eleit"].iloc[0]
print(f"Brasil: candidatas {pct_br_cand:.1f}%, eleitas {pct_br_eleit:.1f}%")

part[["unidade", "total_cand", "pct_fem_cand", "total_eleit", "feminino_eleit", "pct_fem_eleit"]].round(1)

24 de 28 partidos com 100 ou mais candidaturas
Brasil: candidatas 34.9%, eleitas 17.7%


genero,unidade,total_cand,pct_fem_cand,total_eleit,feminino_eleit,pct_fem_eleit
16,PRTB,340,32.4,0,0,NaN
10,PL,509,32.8,99,17,17.2
23,PTB,512,33.0,1,0,NaN
0,AGIR,372,33.1,0,0,NaN
5,NOVO,219,33.3,3,1,NaN
6,PATRIOTA,485,33.4,4,0,NaN
11,PMB,223,33.6,0,0,NaN
1,AVANTE,423,33.8,7,2,NaN
17,PSB,455,33.8,14,2,14.3
24,REPUBLICANOS,531,33.9,40,5,12.5


In [7]:
ok = part[part["eleitas_confiavel"]]
print(f"{len(part)} partidos com 100+ candidaturas; {len(ok)} com 10+ eleitos\n")

resumo = pd.DataFrame({
    f"candidatas, todos ({len(part)})": part["pct_fem_cand"].agg(["min", "max", "median", "std"]),
    f"candidatas, só os {len(ok)}": ok["pct_fem_cand"].agg(["min", "max", "median", "std"]),
    f"eleitas, só os {len(ok)}": ok["pct_fem_eleit"].agg(["min", "max", "median", "std"]),
}).round(1)
resumo

24 partidos com 100+ candidaturas; 12 com 10+ eleitos



,"candidatas, todos (24)","candidatas, só os 12","eleitas, só os 12"
min,32.4,32.8,8.5
max,40.1,40.1,57.1
median,34.1,34.4,16.7
std,2.0,2.4,12.8


In [8]:
COR = {"Candidatas": "#7F7F7F", "Eleitas": "#B5338A"}    # as mesmas cores da questão 2

long = part.melt(id_vars=["unidade", "total_eleit", "feminino_eleit"],
                 value_vars=["pct_fem_cand", "pct_fem_eleit"],
                 var_name="grupo", value_name="pct")
long["grupo"] = long["grupo"].map({"pct_fem_cand": "Candidatas", "pct_fem_eleit": "Eleitas"})

# rótulo "7 de 12" só nas barras de eleitas que aparecem
long["rotulo"] = np.where(
    (long["grupo"] == "Eleitas") & long["pct"].notna(),
    long["feminino_eleit"].astype(str) + " de " + long["total_eleit"].astype(str), "")

titulo = (f"Candidatas: {ok['pct_fem_cand'].min():.0f}% a {ok['pct_fem_cand'].max():.0f}% nos partidos; "
          f"eleitas: {ok['pct_fem_eleit'].min():.0f}% a {ok['pct_fem_eleit'].max():.0f}%"
          f"<br><sup>Deputado federal, 2022. Intervalos calculados entre os partidos com 10 ou mais eleitos.</sup>")

fig_part = px.bar(
    long, x="pct", y="unidade", color="grupo", barmode="group", orientation="h",
    text="rotulo", color_discrete_map=COR,
    category_orders={"unidade": list(part["unidade"])},
    hover_data={"total_eleit": True, "rotulo": False},
    labels={"pct": "% de mulheres", "unidade": "Partido/Federação", "grupo": "", "total_eleit": "Nº de eleitos"},
    title=titulo,
)
fig_part.update_traces(textposition="outside", cliponaxis=False)
fig_part.update_xaxes(range=[0, 70], ticksuffix="%")
fig_part.add_vline(x=pct_br_cand, line_dash="dot", line_color="gray",
                   annotation_text=f"Brasil: candidatas {pct_br_cand:.1f}%", annotation_position="top right")
fig_part.add_vline(x=pct_br_eleit, line_dash="dot", line_color="gray",
                   annotation_text=f"Brasil: eleitas {pct_br_eleit:.1f}%", annotation_position="top left")
fig_part.add_annotation(
    text=f"Partidos sem barra de eleitas elegeram menos de {N_MIN_ELEIT} deputados federais em 2022.",
    xref="paper", yref="paper", x=0, y=-0.1, showarrow=False, xanchor="left", font=dict(size=11))
fig_part.update_layout(height=30 * len(part) + 240, margin=dict(t=130, b=100), title_font_size=16)
fig_part.add_annotation(
    text="Rótulo das barras de eleitas: mulheres eleitas / total de eleitos da unidade. Unidades com pelo menos 10 eleitos.",
    xref="paper", yref="paper", x=0, y=-0.15, showarrow=False, xanchor="left", font=dict(size=11))
fig_part.update_layout(margin=dict(b=140))
fig_part.show()

In [9]:
dep = base[base["cargo"] == "deputado federal"]

pp = cand_eleit(dep, ["ano", "unidade"])
pp = pp[pp["total_cand"] >= N_MIN_CAND].copy()
pp["unidade"] = pp["unidade"].astype(str)
pp["ano"] = pp["ano"].astype(int)

pd.DataFrame({
    "partidos com 100+ candidaturas": pp.groupby("ano").size(),
    "partidos com 10+ eleitos": pp[pp["total_eleit"] >= N_MIN_ELEIT].groupby("ano").size(),
})

,partidos com 100+ candidaturas,partidos com 10+ eleitos
ano,,
2002,13,10
2006,23,11
2010,23,13
2014,26,14
2018,32,15
2022,24,12


In [10]:
long = pp.melt(id_vars=["ano", "unidade", "total_cand", "total_eleit"],
               value_vars=["pct_fem_cand", "pct_fem_eleit"],
               var_name="grupo", value_name="pct")
long["grupo"] = long["grupo"].map({"pct_fem_cand": "Candidatas", "pct_fem_eleit": "Eleitas"})
long = long[~((long["grupo"] == "Eleitas") & (long["total_eleit"] < N_MIN_ELEIT))].dropna(subset=["pct"])
long["pct"] = long["pct"].astype(float)
long["ano"] = long["ano"].astype(str)

fig_box = px.box(
    long, x="ano", y="pct", color="grupo", points="all", color_discrete_map=COR,
    hover_data={"unidade": True, "total_cand": True, "total_eleit": True},
    labels={"pct": "% de mulheres", "ano": "Ano da eleição", "grupo": "",
            "unidade": "Partido", "total_cand": "Nº de candidaturas", "total_eleit": "Nº de eleitos"},
    title=("Os partidos convergiram no percentual de candidatas; as eleitas ficaram bem abaixo <br><sup>Deputado federal. Cada ponto é um partido (em 2022, partido ou federação); candidatas, com 100 ou mais candidaturas; eleitas, com 10 ou mais eleitos.</sup>"),
      )
fig_box.update_yaxes(range=[0, 65], ticksuffix="%")
fig_box.add_hline(y=30, line_dash="dot", line_color="gray", annotation_text="Mínimo legal (30%)",
                  annotation_position="bottom right")
fig_box.update_layout(boxmode="group", height=550, title_font_size=16)
fig_box.show()

In [11]:
disp = long.groupby(["ano", "grupo"])["pct"].agg(
    n="count", mediana="median", minimo="min", maximo="max",
    q25=lambda s: s.quantile(0.25), q75=lambda s: s.quantile(0.75))
disp["amplitude_interquartil"] = disp["q75"] - disp["q25"]
disp.round(1)

n  mediana  minimo  maximo   q25   q75  \
ano  grupo                                                 
2002 Candidatas  13     11.4     5.5    15.9  10.6  13.5   
     Eleitas     10      6.2     0.0    15.4   2.8   8.4   
2006 Candidatas  23     12.6     3.9    21.4  10.4  14.2   
     Eleitas     11      7.3     0.0    22.2   4.3   8.8   
2010 Candidatas  23     22.8    12.7    29.5  19.6  24.3   
     Eleitas     13      7.7     0.0    40.0   4.9   9.3   
2014 Candidatas  26     31.1    25.2    38.6  30.3  32.5   
     Eleitas     14      9.4     0.0    20.0   5.1  12.7   
2018 Candidatas  32     31.7    26.0    41.0  30.6  34.0   
     Eleitas     15     10.7     2.9    50.0   9.7  17.6   
2022 Candidatas  24     34.1    32.4    40.1  33.6  34.6   
     Eleitas     12     16.7     8.5    57.1  13.8  18.6   

                 amplitude_interquartil  
ano  grupo                               
2002 Candidatas                     2.8  
     Eleitas                        5.6  
2006 Candidatas                     3.8  
     Eleitas                        4.5  
2010 Candidatas                     4.7  
     Eleitas                        4.4  
2014 Candidatas                     2.2  
     Eleitas                        7.7  
2018 Candidatas                     3.4  
     Eleitas                        7.9  
2022 Candidatas                     1.1  
     Eleitas                        4.8

### Análise: partidos e federações

Em 2022, 24 das 28 unidades (partidos e federações) tiveram pelo menos 100 candidaturas; PCB, PCO, PSTU e UP ficaram de fora, todos com mais de 30% de candidatas. Entre as 24, a fração de candidatas vai de 32,4% a 40,1% (mediana de 34,1%), perto do piso de 30% da cota. Entre as 12 unidades com pelo menos 10 eleitos, a fração de eleitas vai de 8,5% a 57,1% (mediana de 16,7%), uma faixa bem mais ampla que a das candidatas.

A razão entre a taxa de sucesso de mulheres e a de homens fica abaixo de 1 em 11 das 12 unidades, de 0,18 (PP) a 0,50 (MDB e Federação Brasil da Esperança). A exceção é PSOL Rede, com 1,95. As federações pesam nas conclusões: a Federação Brasil da Esperança (PT, PCdoB e PV) elegeu 21 mulheres entre 81 eleitos (25,9%), PSDB Cidadania 3 entre 18 (16,7%) e PSOL Rede 8 entre 14 (57,1%). As contagens por unidade são pequenas, então cada percentual muda muito com uma ou duas eleitas.

Ao longo do tempo, a mediana de candidatas entre os partidos sobe de 11,4% para 34,1% (2002 a 2022), e a de eleitas de 6,2% para 16,7%. A dispersão entre partidos é medida pela amplitude interquartil, que é menos sensível a outliers que a amplitude total. A dispersão entre os partidos, medida pelo IQR, é maior nas eleitas do que nas candidatas em cinco das seis eleições (em 2022, 4,8 contra 1,1 ponto percentual); a exceção é 2010, com 4,4 contra 4,7. Uma interpretação possível, apenas como hipótese, é que as candidaturas ficam parecidas entre os partidos porque a cota fixa um piso de 30% para todos, enquanto a fração de eleitas depende de quem recebe votos, que varia mais.

O que a análise mostra é descritivo. Ela não separa o efeito do partido do efeito do cargo, da UF ou do tamanho da lista, e não testa causas.

In [12]:
cand_eleit(alvo, ["unidade"]).query("total_cand < 100")[["unidade", "total_cand", "pct_fem_cand"]].round(1)

genero,unidade,total_cand,pct_fem_cand
7,PCB,40,42.5
8,PCO,57,43.9
22,PSTU,38,42.1
27,UP,18,61.1


In [13]:
av = alvo[alvo["valida"]]
t = av.groupby(["unidade", "genero"], observed=True)["eleito"].agg(["sum", "count"]).unstack("genero")

tx = pd.DataFrame({
    "taxa_f": t[("sum", "feminino")] / t[("count", "feminino")] * 100,
    "taxa_m": t[("sum", "masculino")] / t[("count", "masculino")] * 100,
})
tx["razao_f_m"] = tx["taxa_f"] / tx["taxa_m"]
tx.index = tx.index.astype(str)
tx.loc[ok["unidade"]].round(2).sort_values("razao_f_m")

,taxa_f,taxa_m,razao_f_m
unidade,,,
PP,2.47,13.69,0.18
PSD,2.88,14.73,0.20
REPUBLICANOS,2.96,10.45,0.28
PSB,1.38,4.36,0.32
PSDB Cidadania,1.79,5.45,0.33
UNIÃO,5.39,15.92,0.34
PODE,1.23,3.18,0.39
PDT,1.97,4.73,0.42
PL,10.97,25.62,0.43


## 4. Municípios

A cota de gênero vale para as listas, e as câmaras municipais são pequenas (mediana de 9 vereadores), então o percentual de vereadoras por município salta de 11 em 11 pontos. Em vez dele, medimos algo mais robusto: a fração de câmaras em que **nenhuma mulher** foi eleita vereadora, por eleição.

Só entram os municípios que têm pelo menos um eleito registrado na base. Municípios com candidaturas, mas sem nenhum resultado registrado, ficam de fora, e o número de excluídos por ano é mostrado abaixo.

In [14]:
el = base[(base["cargo"] == "vereador") & base["eleito"]].copy()
el["fem"] = (el["genero"] == "feminino").astype(int)

cam_mun = (el.groupby(["ano", "id_municipio"], observed=True)
             .agg(n=("fem", "size"), mulheres=("fem", "sum"))
             .reset_index())
cam_mun["sem_mulher"] = cam_mun["mulheres"] == 0

res = cam_mun.groupby("ano")["sem_mulher"].agg(n_camaras="count", sem_mulher="sum")
res["pct_sem_mulher"] = res["sem_mulher"] / res["n_camaras"] * 100

# municípios excluídos: têm candidaturas a vereador, mas nenhum eleito registrado
cand_mun = base[base["cargo"] == "vereador"].groupby("ano")["id_municipio"].nunique()
res["municipios_com_candidatos"] = cand_mun
res["excluidos"] = res["municipios_com_candidatos"] - res["n_camaras"]
res.round(1)

,n_camaras,sem_mulher,pct_sem_mulher,municipios_com_candidatos,excluidos
ano,,,,,
2004,5559,1636,29.4,5561,2
2008,5550,1663,30.0,5560,10
2012,5567,1333,23.9,5568,1
2016,5563,1294,23.3,5568,5
2020,5564,930,16.7,5567,3
2024,5514,728,13.2,5569,55


In [15]:
v24 = base[(base["ano"] == 2024) & (base["cargo"] == "vereador")]

com_eleito = set(v24.loc[v24["eleito"], "id_municipio"])
sem_eleito = set(v24["id_municipio"]) - com_eleito

print(len(sem_eleito))
print(v24[v24["id_municipio"].isin(sem_eleito)]["resultado"].value_counts(dropna=False))

55
resultado
None    5792
Name: count, dtype: int64


In [16]:
c9 = cam_mun[cam_mun["n"] >= 9]
res9 = c9.groupby("ano")["sem_mulher"].agg(n_camaras="count", sem_mulher="sum")
res9["pct_sem_mulher"] = res9["sem_mulher"] / res9["n_camaras"] * 100

comp_cam = res[["pct_sem_mulher"]].join(res9[["pct_sem_mulher"]], lsuffix="_todas", rsuffix="_9mais")
comp_cam["dif_pp"] = comp_cam["pct_sem_mulher_9mais"] - comp_cam["pct_sem_mulher_todas"]
comp_cam.round(1)

,pct_sem_mulher_todas,pct_sem_mulher_9mais,dif_pp
ano,,,
2004,29.4,29.4,0.0
2008,30.0,30.0,-0.0
2012,23.9,23.9,-0.0
2016,23.3,23.3,0.0
2020,16.7,16.7,0.0
2024,13.2,13.2,-0.0


In [17]:
cam_mun["porte"] = np.where(cam_mun["n"] <= 9, "até 9 vereadores", "10 ou mais")
por_porte = (cam_mun.groupby(["ano", "porte"])["sem_mulher"]
               .agg(n_camaras="count", sem_mulher="sum"))
por_porte["pct_sem_mulher"] = por_porte["sem_mulher"] / por_porte["n_camaras"] * 100
por_porte.round(1).unstack("porte")

n_camaras                  sem_mulher                  pct_sem_mulher  \
porte 10 ou mais até 9 vereadores 10 ou mais até 9 vereadores     10 ou mais   
ano                                                                            
2004         568             4991        153             1483           26.9   
2008         600             4950        189             1474           31.5   
2012        1828             3739        355              978           19.4   
2016        1921             3642        372              922           19.4   
2020        1963             3601        269              661           13.7   
2024        1942             3572        201              527           10.4   

                        
porte até 9 vereadores  
ano                     
2004              29.7  
2008              29.8  
2012              26.2  
2016              25.3  
2020              18.4  
2024              14.8

In [18]:
exc = res["excluidos"]
titulo_cam = (f"A fração de câmaras municipais sem nenhuma vereadora variou entre "
              f"{res.loc[2004, 'pct_sem_mulher']:.0f}% em 2004 e {res.loc[2024, 'pct_sem_mulher']:.0f}% em 2024"
              "<br><sup>Municípios com pelo menos um vereador eleito registrado na base.</sup>")

res2 = res.reset_index()
res2["ano"] = res2["ano"].astype(int)
res2["pct_sem_mulher"] = res2["pct_sem_mulher"].astype(float)

fig_cam = px.line(
    res2, x="ano", y="pct_sem_mulher", markers=True,
    hover_data={"n_camaras": True, "sem_mulher": True, "excluidos": True},
    labels={"pct_sem_mulher": "% de câmaras sem nenhuma vereadora", "ano": "Ano da eleição",
            "n_camaras": "Nº de câmaras", "sem_mulher": "Câmaras sem vereadora", "excluidos": "Municípios excluídos"},
    title=titulo_cam,
)
fig_cam.update_traces(line_color="#B5338A", marker_color="#B5338A")
fig_cam.update_xaxes(tickmode="linear", dtick=4, tick0=2004)
fig_cam.update_yaxes(range=[0, 40], ticksuffix="%")
fig_cam.add_annotation(
    text=f"Nota: excluídos de {exc.min()} a {exc.max()} municípios por eleição, sem resultado registrado na base.",
    xref="paper", yref="paper", x=0, y=-0.22, showarrow=False, xanchor="left", font=dict(size=11))
fig_cam.update_layout(height=480, margin=dict(t=110, b=110), title_font_size=16)
fig_cam.show()

## 4. Estados

In [19]:
import json, requests, os

os.makedirs("../data/geo", exist_ok=True)
FMT = "formato=application/vnd.geo+json"
BASE = "https://servicodados.ibge.gov.br/api/v3/malhas"

geo = None

u1 = f"{BASE}/paises/BR?intrarregiao=UF&qualidade=minima&{FMT}"
r = requests.get(u1, timeout=60)
g = r.json()
if len(g.get("features", [])) == 27:
    geo = g

with open("../data/geo/ufs.geojson", "w", encoding="utf-8") as f:
    json.dump(geo, f)

print(len(geo["features"]))
print(geo["features"][0]["properties"])

27
{'codarea': '11'}


In [21]:
def area(ring):
    return sum(x1*y2 - x2*y1 for (x1, y1), (x2, y2) in zip(ring, ring[1:])) / 2
    
def orienta(poly):
    out = []
    for i, ring in enumerate(poly):
        ring = [tuple(p[:2]) for p in ring]
        if (area(ring) > 0) == (i == 0):   # externo -> horário; buracos -> anti-horário
            ring = ring[::-1]
        out.append(ring)
    return out

for f in geo["features"]:
    g = f["geometry"]
    if g["type"] == "Polygon":
        g["coordinates"] = orienta(g["coordinates"])
    else:
        g["coordinates"] = [orienta(p) for p in g["coordinates"]]

with open("../data/geo/ufs.geojson", "w", encoding="utf-8") as fh:
    json.dump(geo, fh)

In [22]:
base["cargo"].value_counts()

cargo
vereador              2576725
prefeito                98682
deputado estadual       89449
deputado federal        41629
deputado distrital       4802
senador                  1567
governador               1166
presidente                 62
Name: count, dtype: int64

In [23]:
CODIGO_UF = {"RO":"11","AC":"12","AM":"13","RR":"14","PA":"15","AP":"16","TO":"17",
             "MA":"21","PI":"22","CE":"23","RN":"24","PB":"25","PE":"26","AL":"27","SE":"28","BA":"29",
             "MG":"31","ES":"32","RJ":"33","SP":"35","PR":"41","SC":"42","RS":"43",
             "MS":"50","MT":"51","GO":"52","DF":"53"}

est = base[base["cargo"].isin(["deputado estadual", "deputado distrital"])]
mapa = cand_eleit(est, ["ano", "sigla_uf"])
mapa["cod"] = mapa["sigla_uf"].astype(str).map(CODIGO_UF)

print(mapa.groupby("ano").size())
print(mapa["total_eleit"].describe())
print(mapa["cod"].isna().sum())

ano
2002    27
2006    27
2010    27
2014    27
2018    27
2022    27
dtype: int64
count    162.000000
mean      39.222222
std       18.759014
min       24.000000
25%       24.000000
50%       30.000000
75%       49.000000
max       94.000000
Name: total_eleit, dtype: float64
0


In [24]:
import plotly.graph_objects as go

ANO = 2022
m22 = mapa[mapa["ano"].astype(int) == ANO].copy()
m22["pct_fem_eleit"] = m22["pct_fem_eleit"].astype(float)
m22["eleitas"] = m22["feminino_eleit"].astype(int)
m22["eleitos"] = m22["total_eleit"].astype(int)

# posição de cada sigla: centro do maior polígono de cada estado
def area(ring):
    return sum(x1*y2 - x2*y1 for (x1, y1), (x2, y2) in zip(ring, ring[1:])) / 2

def centro(geom):
    polis = [geom["coordinates"]] if geom["type"] == "Polygon" else geom["coordinates"]
    ring = max((p[0] for p in polis), key=lambda r: abs(area([tuple(q[:2]) for q in r])))
    ring = [tuple(q[:2]) for q in ring]
    a = area(ring)
    cx = sum((x1 + x2) * (x1*y2 - x2*y1) for (x1, y1), (x2, y2) in zip(ring, ring[1:])) / (6 * a)
    cy = sum((y1 + y2) * (x1*y2 - x2*y1) for (x1, y1), (x2, y2) in zip(ring, ring[1:])) / (6 * a)
    return cx, cy

pos = {f["properties"]["codarea"]: centro(f["geometry"]) for f in geo["features"]}
m22["lon"] = m22["cod"].map(lambda c: pos[c][0])
m22["lat"] = m22["cod"].map(lambda c: pos[c][1])

mx = m22.loc[m22["pct_fem_eleit"].idxmax()]
mn = m22.loc[m22["pct_fem_eleit"].idxmin()]
titulo_mapa = (f"Em {ANO}, a fração de mulheres entre os deputados estaduais eleitos vai de "
               f"{mn['pct_fem_eleit']:.0f}% ({mn['sigla_uf']}) a {mx['pct_fem_eleit']:.0f}% ({mx['sigla_uf']})"
               "<br><sup>Deputados estaduais e, no DF, distritais.</sup>")

fig_mapa = px.choropleth(
    m22, geojson=geo, locations="cod", featureidkey="properties.codarea",
    color="pct_fem_eleit", range_color=(0, float(np.ceil(m22["pct_fem_eleit"].max() / 5) * 5)),
    color_continuous_scale=["#F6E3EF", "#B5338A"],
    hover_name="sigla_uf",
    hover_data={"cod": False, "pct_fem_eleit": ":.1f", "eleitas": True, "eleitos": True},
    labels={"pct_fem_eleit": "% de mulheres<br>entre os eleitos", "eleitas": "Mulheres eleitas", "eleitos": "Total de eleitos"},
    title=titulo_mapa,
)
fig_mapa.update_traces(marker_line_color="white", marker_line_width=0.8)
fig_mapa.add_trace(go.Scattergeo(
    lon=m22["lon"], lat=m22["lat"], text=m22["sigla_uf"], mode="text",
    textfont=dict(size=11, color="#222222"), hoverinfo="skip", showlegend=False))
fig_mapa.update_geos(fitbounds="locations", visible=False)
fig_mapa.update_layout(height=650, margin=dict(t=100, b=20, l=0, r=0), title_font_size=16)
fig_mapa.show()

In [25]:
tab = (m22[["sigla_uf", "eleitas", "eleitos", "pct_fem_eleit"]]
         .sort_values("pct_fem_eleit", ascending=False)
         .round({"pct_fem_eleit": 1}))
display(tab.head(5))
display(tab.tail(5))
print("mediana:", round(m22["pct_fem_eleit"].median(), 1))

genero,sigla_uf,eleitas,eleitos,pct_fem_eleit
138,AP,7,24,29.2
144,MA,12,42,28.6
160,SP,25,94,26.6
136,AL,6,27,22.2
153,RJ,15,70,21.4


genero,sigla_uf,eleitas,eleitos,pct_fem_eleit
150,PE,6,49,12.2
143,GO,4,41,9.8
146,MS,2,24,8.3
158,SC,3,40,7.5
147,MT,1,24,4.2


mediana: 18.5


In [26]:
p = m22["eleitas"].sum() / m22["eleitos"].sum()
m22["esperado"] = m22["eleitos"] * p
m22["z"] = (m22["eleitas"] - m22["esperado"]) / np.sqrt(m22["eleitos"] * p * (1 - p))

print("fração nacional:", round(p * 100, 1))
print("UFs com |z| > 2:", int((m22["z"].abs() > 2).sum()), "de", len(m22))
m22.sort_values("z")[["sigla_uf", "eleitas", "eleitos", "esperado", "z"]].round(1)

fração nacional: 17.9
UFs com |z| > 2: 1 de 27


genero,sigla_uf,eleitas,eleitos,esperado,z
147,MT,1,24,4.3,-1.8
158,SC,3,40,7.2,-1.7
143,GO,4,41,7.4,-1.4
146,MS,2,24,4.3,-1.2
139,BA,8,63,11.3,-1.1
150,PE,6,49,8.8,-1.0
161,TO,3,24,4.3,-0.7
135,AC,3,24,4.3,-0.7
142,ES,4,30,5.4,-0.7
151,PI,4,30,5.4,-0.7


In [27]:
ve = base[base["cargo"] == "vereador"]
mv = cand_eleit(ve, ["ano", "sigla_uf"])
mv["ano"] = mv["ano"].astype(int)
mv["sigla_uf"] = mv["sigla_uf"].astype(str)

print(mv.groupby("ano").size())
print(mv["total_eleit"].describe())
print(sorted(set(CODIGO_UF) - set(mv["sigla_uf"])))

ano
2004    26
2008    26
2012    26
2016    26
2020    26
2024    26
dtype: int64
count     156.000000
mean     2144.730769
std      1968.753378
min       139.000000
25%       810.250000
50%      1638.000000
75%      2495.500000
max      8481.000000
Name: total_eleit, dtype: float64
['DF']


In [29]:
import plotly.graph_objects as go

mv["pct_fem_eleit"] = mv["pct_fem_eleit"].astype(float)
piv = mv.pivot(index="sigla_uf", columns="ano", values="pct_fem_eleit")
piv = piv.sort_values(piv.columns[-1], ascending=False)


d = piv[[2004, 2024]].sort_values(2024).astype(float)

# UFs com poucos vereadores eleitos em alguma eleição ganham asterisco
pequenas = set(mv.groupby("sigla_uf")["total_eleit"].min().loc[lambda s: s < 250].index)
rot = [f"{u}*" if u in pequenas else u for u in d.index]

n_sobem = int((d[2024] > d[2004]).sum())
titulo_dumb = (f"A fração de vereadoras eleitas aumentou em {n_sobem} das {len(d)} UFs entre 2004 e 2024"
               "<br><sup>Linhas pontilhadas: média nacional. UFs ordenadas pela fração em 2024. O DF não tem vereadores.</sup>")

fig_dumb = go.Figure()
for r, a, b in zip(rot, d[2004], d[2024]):
    fig_dumb.add_trace(go.Scatter(x=[a, b], y=[r, r], mode="lines",
                                  line=dict(color="#C9C9C9", width=3), hoverinfo="skip", showlegend=False))
fig_dumb.add_trace(go.Scatter(x=d[2004], y=rot, mode="markers", name="2004",
                              marker=dict(color="#8C8C8C", size=10),
                              hovertemplate="%{y}, 2004: %{x:.1f}%<extra></extra>"))
fig_dumb.add_trace(go.Scatter(x=d[2024], y=rot, mode="markers", name="2024",
                              marker=dict(color="#B5338A", size=10),
                              hovertemplate="%{y}, 2024: %{x:.1f}%<extra></extra>"))

fig_dumb.add_vline(x=nac.loc[2004], line_dash="dot", line_color="#8C8C8C")
fig_dumb.add_vline(x=nac.loc[2024], line_dash="dot", line_color="#B5338A")

fig_dumb.update_yaxes(categoryorder="array", categoryarray=rot, title="UF")
fig_dumb.update_xaxes(ticksuffix="%", range=[0, 30], title="% de vereadoras entre os eleitos")
fig_dumb.add_annotation(
    text="* UF com menos de 250 vereadores eleitos em pelo menos uma eleição (RR, AP, AC): a fração oscila mais.",
    xref="paper", yref="paper", x=0, y=-0.12, showarrow=False, xanchor="left", font=dict(size=11))
fig_dumb.update_layout(height=700, margin=dict(t=100, b=100), title=titulo_dumb, title_font_size=16,
                       legend=dict(orientation="h", y=1.02, x=1, xanchor="right"))
fig_dumb.show()

## Conclusão
1. Em 2022, a fração de mulheres entre os deputados estaduais eleitos (distritais, no DF) variou de 4,2% em Mato Grosso (1 de 24) a 29,2% no Amapá (7 de 24), com mediana de 18,5% entre as 27 UFs; a fração nacional foi de 17,9%. Bancadas pequenas amplificam as diferenças: em estados com 24 cadeiras, cada cadeira equivale a 4,2 pontos percentuais. Comparando cada UF com a fração nacional, considerando o tamanho da bancada, apenas São Paulo (25 de 94) se afasta em mais de dois desvios.

2. A fração de vereadoras entre os eleitos aumentou entre 2004 e 2024 nas 26 UFs com vereadores (o DF não tem). No Brasil, ela ficou entre 12,5% e 13,5% de 2004 a 2016 e subiu para 16,1% em 2020 e 18,2% em 2024. As maiores altas foram em Roraima (+11,9 pontos), Piauí (+9,7) e Amapá (+9,6), e a menor foi no Rio de Janeiro (+0,4). Roraima, Amapá e Acre têm poucos vereadores eleitos (de 139 a 241 por eleição), então suas frações oscilam mais de uma eleição para outra. A figura é descritiva e não testa explicações para as diferenças entre estados.